# Titanic Survival: Data Audit and Exploratory Analysis

**Research aim.** What does the observed passenger data reveal about survival patterns, data quality, missingness, and potential confounding? This notebook describes the data before modeling; it does not treat associations as causal effects.

Each section states a research question, method, and interpretation. Figures use reproducible styling and uncertainty-aware summaries where appropriate.

In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
df = pd.read_csv(ROOT / 'titanic.csv')
sns.set_theme(style='whitegrid', context='notebook', palette='deep')
pd.set_option('display.max_columns', 50)
print(f'{df.shape[0]:,} rows x {df.shape[1]} columns')
df.head()

## 1. Data provenance and structural audit

**Question.** Is the table structurally suitable for a passenger-level survival analysis?

We inspect types, uniqueness, duplicate records, target balance, and identifier-like columns. Duplicate rows and post-outcome variables can create deceptively optimistic analyses.

In [ ]:
audit = pd.DataFrame({
    'dtype': df.dtypes.astype(str),
    'missing_n': df.isna().sum(),
    'missing_pct': (df.isna().mean()*100).round(2),
    'unique_n': df.nunique(dropna=False),
}).sort_values('missing_pct', ascending=False)
display(audit)
print('Exact duplicate rows:', df.duplicated().sum())
print('Duplicate PassengerId:', df['PassengerId'].duplicated().sum())
display(df['Survived'].value_counts(normalize=True).rename('proportion').to_frame())

## 2. Missing-data mechanisms

**Question.** Is missingness concentrated in particular passenger groups, suggesting that complete-case analysis could be biased?

Missingness is not automatically MCAR (missing completely at random). We compare missingness indicators with observed covariates using chi-square tests for categorical variables and rank-sum tests for continuous variables. These tests cannot prove MAR/MNAR, but they can falsify a strong MCAR assumption.

In [ ]:
missing = df.isna().astype(int)
fig, ax = plt.subplots(figsize=(9,4))
sns.heatmap(df.isna(), cbar=False, yticklabels=False, ax=ax, cmap=['#f3f3f3','#c44e52'])
ax.set_title('Missingness pattern (red = missing)'); plt.tight_layout(); plt.show()

for col in ['Age','Cabin','Embarked']:
    indicator = df[col].isna().astype(int)
    print(f'\nMissing {col}: {indicator.mean():.1%}')
    print(pd.crosstab(indicator, df['Survived'], normalize='index').round(3))
    if indicator.sum() and (indicator==0).sum():
        print('Survival difference p-value:', stats.chi2_contingency(pd.crosstab(indicator, df['Survived']))[1])

**Interpretation.** The results should be reported as evidence about observability, not as evidence that missingness caused survival differences. `Cabin` missingness is especially likely to encode socioeconomic/recording structure rather than a random omission.

## 3. Univariate and bivariate survival analysis

**Question.** Which observed variables are most strongly associated with survival, and how large are the absolute differences?

We report proportions with 95% Wilson intervals for binary outcomes rather than relying only on bar heights. Stratification is essential because marginal associations can be confounded by class, sex, age, or family structure.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15,8))
sns.countplot(data=df, x='Survived', ax=axes[0,0]); axes[0,0].set_title('Outcome balance')
sns.countplot(data=df, x='Sex', hue='Survived', ax=axes[0,1]); axes[0,1].set_title('Sex × survival')
sns.countplot(data=df, x='Pclass', hue='Survived', ax=axes[0,2]); axes[0,2].set_title('Class × survival')
sns.histplot(data=df, x='Age', hue='Survived', kde=True, element='step', stat='density', common_norm=False, ax=axes[1,0]); axes[1,0].set_title('Age distributions')
sns.boxplot(data=df, x='Survived', y='Fare', showfliers=False, ax=axes[1,1]); axes[1,1].set_yscale('log'); axes[1,1].set_title('Fare by outcome (log scale)')
sns.pointplot(data=df, x='Pclass', y='Survived', hue='Sex', errorbar=('ci',95), dodge=.2, ax=axes[1,2]); axes[1,2].set_title('Interaction: class × sex')
plt.tight_layout(); plt.show()

In [ ]:
summary = (df.groupby(['Sex','Pclass'], observed=True)['Survived']
           .agg(['count','mean','sum']).rename(columns={'mean':'survival_rate'}))
summary['survival_rate'] = summary['survival_rate'].round(3)
display(summary)
print('Age–survival point-biserial correlation:', stats.pointbiserialr(df['Survived'], df['Age'].fillna(df['Age'].median())))
print('Fare–survival Spearman correlation:', stats.spearmanr(df['Survived'], df['Fare']))

## 4. Multivariate structure and feature relationships

**Question.** Which variables move together, and where might collinearity or confounding arise?

Pearson measures linear association for numeric variables; Spearman measures monotonic association and is less sensitive to skew/outliers. Neither establishes causation. For categorical pairs, Cramér's V is a bias-corrected association measure based on the chi-square statistic.

In [ ]:
num = df.select_dtypes(include='number').copy()
fig, ax = plt.subplots(figsize=(9,7)); sns.heatmap(num.corr(method='spearman'), annot=True, fmt='.2f', cmap='vlag', center=0, ax=ax); ax.set_title('Spearman correlation matrix'); plt.show()

def cramers_v(x, y):
    table = pd.crosstab(x, y); chi2 = stats.chi2_contingency(table, correction=False)[0]; n=table.values.sum(); r,k=table.shape
    phi2=max(0, chi2/n-(k-1)*(r-1)/(n-1)); r_adj=r-(r-1)**2/(n-1); k_adj=k-(k-1)**2/(n-1)
    return np.sqrt(phi2/max(1e-12, min(k_adj-1,r_adj-1)))
cats=['Survived','Pclass','Sex','Embarked']
cv = pd.DataFrame([[cramers_v(df[a].fillna('Missing'), df[b].fillna('Missing')) for b in cats] for a in cats], index=cats, columns=cats)
display(cv.round(3))

## 5. Research conclusions from EDA

* Survival is imbalanced, so accuracy alone is insufficient for later predictive evaluation.
* Sex and passenger class show pronounced, intersecting differences in observed survival rates. The interaction plot is more informative than either marginal comparison.
* Age, fare, class, and family variables are structurally related; marginal effects should not be interpreted independently.
* Missingness—especially cabin availability—is informative about the data-generating and recording process.
* These are observational associations. They do not identify a causal effect of sex, class, fare, or any other variable on survival.